In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
import openai
import psycopg2
import time

In [2]:
load_dotenv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\.env")
openai.api_key=os.environ.get("OPENAI_API_KEY")

In [48]:
conn = psycopg2.connect(
    host=os.environ.get("PGHOST"),
    port=os.environ.get("PGPORT"),
    user=os.environ.get("PGUSER"),
    password=os.environ.get("PGPASSWORD"),
    database=os.environ.get("PGDATABASE"),
)

In [40]:
def translate(enUS, ptBR):
    response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo",
      messages=[
        {"role": "system", "content": "You are a translation engine that, in the contex of comex for shipment of goods, can only translate text from english or brasilian portuguese to spanish and cannot interpret it. If you cannot translate it, retur the same value."},
        {"role": "user", "content": f"en-US: {enUS}\npt-BR: {ptBR}"},
      ],
      temperature=0,
      n=1,
    )
    translation = response.choices[0].message.content.strip()
    if (translation[-1] == ".") and (enUS[-1] != "."):
      translation = translation[:-1]
    if translation.startswith("es: "):
       translation = translation[4:]
    if translation.startswith("es-ES: "):
       translation = translation[7:]

    return translation

translate("Demurrage", "Demurrage")

'Demora en puerto'

In [43]:
def load_data():
    if os.path.exists("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv"):
        data = pd.read_csv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv")
    else:
        with conn.cursor() as cur:
            cur.execute("SELECT dsmsgkey, dsenus, dsptbr, dseses FROM smart.i18n where dtdeleted is null")
            data = pd.DataFrame(cur.fetchall(), columns=["dsmsgkey", "dsenus", "dsptbr", "dseses"])
    return data

data = load_data()
data

,dsmsgkey,dsenus,dsptbr,dseses
0,Label.notes,Notes,Notas / Observações,Notas
1,ActionGroupTypes.AIR,Air,Aéreo,Aire
2,ActionGroupTypes.QUOTATION,Quotation,Cotação,Cotización
3,ActionGroupTypes.SEA,Sea,Marítimo,Mar
4,Action.name.ADDADDRESS,Add Address,Adicionar Endereço,Agregar dirección.
...,...,...,...,...
4970,rBuilder.i18nLbNF,NF,NF,NF
4971,rBuilder.i18nLbPPCC,PP/CC,PP/CC,PP/CC
4972,rBuilder.i18nLbQuantityTon,Qtd TON,Qtd TON,Qtd TON
4973,Shortcut.Accesskey.Yes,Y,S,S


In [10]:
def save_csv(data):
    data.to_csv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv", index=False)

In [11]:
def save_data(data):
    save_csv(data)
    with conn.cursor() as cur:
        for _, row in data.iterrows():
            cur.execute(
                "UPDATE smart.i18n SET dseses = %s WHERE dsmsgkey = %s",
                (row["dseses"], row["dsmsgkey"]),
            )
        conn.commit()

In [ ]:
data = load_data()
data

In [12]:
save_csv(data)

In [ ]:
translate("Air")

In [44]:
# seta as traduções pra nulo
data["dseses"] = None

In [47]:
for idx, row in data.iterrows():
    if pd.isna(row["dseses"]):
        try:
            translation = translate(row["dsenus"], row["dsptbr"])
            data.loc[idx, "dseses"] = translation
            save_csv(data)
            print(f"Translated {row['dsmsgkey']}")
            time.sleep(3)
        except:
            print(f"Error translating {row['dsmsgkey']}")
            time.sleep(3)


Translated Caption.toPay
Translated err.Process has invoice. Unable to delete
Translated Label.addPort
Translated Label.clickToReleaseDraft
Translated Label.dsGood
Translated Label.obOriginInput
Translated Product.nmProduct.WhiteLabel
Translated Profiles.button.addProfile
Translated rBuilder.i18nLbPhoneCnee
Translated RoleActionsGroup.name.PAYMENT
Translated SmartPopover.definition.cnee
Translated trackingStatusTypes.BLOCKED
Translated Warning.NoMasterVolumes
Translated Widget.desc.table-processes-bellow-minimun-profit
Translated AirlineForm.placeholder.cdIcao


In [46]:
data["dseses"].isnull().sum()

15

In [33]:
data

,dsmsgkey,dsenus,dsptbr,dseses
0,Action.name.DELETEPROCESSGROUNDNATIONAL,Delete Process Ground National,Remover Processo Rodoviário Doméstico,Remover Proceso Nacional Terrestre
1,AirlineForm.placeholder.cdAirline,eg. G3,ex. G3,ej. G3
2,AirlineForm.placeholder.cdIcao,eg. GLO,ex. GLO,ej. GLO
3,CalculationTypes.PER_CNTR,Per CNTR,Por CNTR,Por CNTR
4,Caption.cdCreditDebit,C / D,C / D,C / D
...,...,...,...,...
63,rBuilder.i18nLbNF,NF,NF,NF
64,rBuilder.i18nLbPPCC,PP/CC,PP/CC,PP/CC
65,rBuilder.i18nLbQuantityTon,Qtd TON,Qtd TON,Qtd TON
66,Shortcut.Accesskey.Yes,Y,S,S


In [49]:
save_csv(data)

In [50]:
save_data(data)

In [51]:
conn.close()
